# Data and provenance

This notebook reads the authorized analysis workbook directly, validates the two observation units, and audits types, missingness, duplicates, transformations, and temporal coverage. It does not reconstruct the unavailable show-week microdata.

In [1]:
from pathlib import Path
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from broadway_research.ingest import load

# Public clones must supply the authorized six-sheet workbook separately.
# Set BROADWAY_INPUT_DIR to the directory containing that workbook.
input_value = os.environ.get("BROADWAY_INPUT_DIR")
if not input_value:
    raise FileNotFoundError("Set BROADWAY_INPUT_DIR to the authorized workbook directory.")
input_dir = Path(input_value)
if not input_dir.exists() or not list(input_dir.glob("*.xlsx")):
    raise FileNotFoundError(
        "Authorized source workbook not found. Set BROADWAY_INPUT_DIR to its containing directory. "
        "The workbook is intentionally excluded from the public repository."
    )

production, market = load(input_dir)
print(f"Loaded authorized inputs: {production.shape[0]} production aggregates and {market.shape[0]} market weeks")

Loaded authorized inputs: 67 production aggregates and 501 market weeks


In [2]:
workbook = next(input_dir.glob("*.xlsx"))
xls = pd.ExcelFile(workbook)
sheet_inventory = []
for sheet_number, sheet in enumerate(xls.sheet_names, start=1):
    frame = pd.read_excel(workbook, sheet_name=sheet)
    if "Show" in frame.columns:
        role = "production analysis table"
    elif "week_ending" in frame.columns:
        role = "market analysis table"
    else:
        role = "supporting result or method table"
    sheet_inventory.append({"sheet_number": sheet_number, "public_role": role, "rows": len(frame), "columns": len(frame.columns)})
pd.DataFrame(sheet_inventory)

,sheet_number,public_role,rows,columns
0,1,supporting result or method table,86,10
1,2,supporting result or method table,11,10
2,3,supporting result or method table,8,6
3,4,production analysis table,67,19
4,5,market analysis table,501,21
5,6,supporting result or method table,10,2


In [3]:
assert production.shape == (67, 19)
assert market.shape == (501, 21)
assert production["Show"].is_unique
assert market["week_ending"].is_unique
assert (production["Avg_Gross_USD"] > 0).all()
assert (market["market_gross_usd"] > 0).all()

transform_checks = pd.Series({
    "max_abs_log_gross_error": np.max(np.abs(np.log(production["Avg_Gross_USD"]) - production["log_gross"])),
    "min_back_transformed_seats": np.exp(production["log_seats"]).min(),
    "max_back_transformed_seats": np.exp(production["log_seats"]).max(),
    "production_duplicate_rows": production.duplicated().sum(),
    "market_duplicate_dates": market["week_ending"].duplicated().sum(),
})
transform_checks

max_abs_log_gross_error          0.0
min_back_transformed_seats     583.0
max_back_transformed_seats    1874.0
production_duplicate_rows        0.0
market_duplicate_dates           0.0
dtype: float64

In [4]:
missing = pd.concat({
    "production": production.isna().sum(),
    "market": market.isna().sum(),
}, axis=0).rename("missing_count").to_frame()
missing.query("missing_count > 0")

missing_count
market unemployment_rate_pct                   244
       nyc_visitors_millions_annual             21
       holiday_name                            399

In [5]:
coverage = pd.Series({
    "first_market_week": market["week_ending"].min().date(),
    "last_market_week": market["week_ending"].max().date(),
    "observed_weeks": len(market),
    "calendar_gaps_over_14_days": int(market.sort_values("week_ending")["week_ending"].diff().dt.days.gt(14).sum()),
    "unemployment_missing_count": int(market["unemployment_rate_pct"].isna().sum()),
    "unemployment_missing_rate": market["unemployment_rate_pct"].isna().mean(),
})
coverage

first_market_week             2015-05-31
last_market_week              2026-05-24
observed_weeks                       501
calendar_gaps_over_14_days             1
unemployment_missing_count           244
unemployment_missing_rate       0.487026
dtype: object

The 67 production rows are title-level aggregates. Their source paper describes approximately 4,972 underlying show-week observations, but those rows and the complete selection trail are not supplied. The 501 market rows are separate calendar-week observations and must not be merged row-for-row with the production table.